# 🔬 Análise de Dados: Prevenção e Mortalidade do Câncer de Colo do Útero no Brasil## Versão Profissional — Visualizações Interativas com Plotly**Autor:** Iago Almeida  **Fontes:** DATASUS (SIM, SISCAN, SI-PNI, IBGE)  **Período:** 2013–2024  ---> Este notebook realiza a extração automatizada de dados do DATASUS, engenharia de atributos,> análise estatística avançada e visualização interativa para investigar a eficácia das> políticas públicas de rastreamento (Papanicolau) e prevenção (Vacinação HPV) do câncer> de colo do útero nos 27 estados brasileiros.

## 1. Setup & Configurações

In [30]:
# ==============================================================================# 1.1 — Imports e Configurações Globais# ==============================================================================import warningswarnings.filterwarnings('ignore')# Coreimport pandas as pdimport numpy as npimport io, ast, re, time, os, json# Web Scrapingimport requestsfrom requests.adapters import HTTPAdapterfrom urllib3.util.retry import Retry# Estatística & MLimport scipy.stats as statsfrom sklearn.linear_model import LinearRegressionfrom sklearn.preprocessing import StandardScalerfrom sklearn.cluster import KMeansfrom sklearn.decomposition import PCAfrom sklearn.metrics import silhouette_scorefrom statsmodels.tsa.seasonal import seasonal_decompose# Visualização Estáticaimport matplotlib.pyplot as pltimport seaborn as sns# Visualização Interativaimport plotly.express as pximport plotly.graph_objects as gofrom plotly.subplots import make_subplotsimport plotly.io as pio# Configssns.set(style="whitegrid")plt.rcParams['figure.dpi'] = 100# Tema Plotly profissional darkPLOTLY_TEMPLATE = 'plotly_dark'pio.templates.default = PLOTLY_TEMPLATE# Paleta de cores institucionalCORES = {    'primaria': '#E63946',    # Vermelho    'secundaria': '#457B9D',  # Azul    'terciaria': '#2A9D8F',   # Verde-água    'alerta': '#F4A261',      # Laranja    'fundo': '#1A1A2E',       # Fundo escuro    'texto': '#E0E0E0',       # Texto claro}PALETA_REGIOES = {    'Norte': '#E63946', 'Nordeste': '#F4A261', 'Sudeste': '#457B9D',    'Sul': '#2A9D8F', 'Centro-Oeste': '#A855F7'}DATA_DIR = 'data'os.makedirs(DATA_DIR, exist_ok=True)print("✅ Ambiente configurado com sucesso!")print(f"📁 Diretório de cache: {DATA_DIR}/")

## 2. Engenharia de Dados — Extração do DATASUS com Cache Local

### 2.1 Funções Auxiliares de Extração> Sistema robusto de extração com **cache local em CSV**. Se os dados já foram baixados,> o notebook carrega do disco instantaneamente, evitando re-scraping do DATASUS.

In [31]:
# ==============================================================================# 2.1 — Funções Auxiliares de Extração e Cache# ==============================================================================def carregar_ou_extrair(nome_cache, func_extração, *args, **kwargs):    """Tenta carregar CSV do cache; se não existir, executa a extração."""    caminho = os.path.join(DATA_DIR, f'{nome_cache}.csv')    if os.path.exists(caminho):        print(f"📂 Carregando cache: {caminho}")        return pd.read_csv(caminho)    else:        print(f"🌐 Cache não encontrado. Extraindo dados do DATASUS...")        df = func_extração(*args, **kwargs)        df.to_csv(caminho, index=False)        print(f"💾 Dados salvos em: {caminho}")        return dfdef salvar_cache(df, nome):    caminho = os.path.join(DATA_DIR, f'{nome}.csv')    df.to_csv(caminho, index=False)    print(f"💾 Cache salvo: {caminho}")def extrair_tabnet_html(url, payload_string, headers):    """Extrai tabela HTML do TabNet (SIM, IBGE)."""    response = requests.post(url, data=payload_string, headers=headers)    response.raise_for_status()    response.encoding = 'ISO-8859-1'    lista_de_tabelas = pd.read_html(io.StringIO(response.text))    df = lista_de_tabelas[0]    if isinstance(df.columns, pd.MultiIndex):        df.columns = df.columns.droplevel(0)    df = df.rename(columns={df.columns[0]: 'Unidade da Federação'})    colunas_para_manter = [        col for col in df.columns        if col == 'Unidade da Federação' or str(col).isdigit()    ]    df = df[colunas_para_manter]    if str(df.iloc[0, 0]).upper() == 'TOTAL':        df = df.iloc[1:].reset_index(drop=True)    split_data = df['Unidade da Federação'].str.split(' ', n=1, expand=True)    df['id_uf'] = split_data[0]    df['uf_nome'] = split_data[1]    df = df.drop(columns=['Unidade da Federação'])    colunas_ordenadas = ['id_uf', 'uf_nome'] + [c for c in df.columns if c not in ['id_uf', 'uf_nome']]    return df[colunas_ordenadas]def extrair_tabnet_js(url, payload, headers):    """Extrai dados de resposta JavaScript do TabNet (SISCAN, PNI)."""    response = requests.post(url, data=payload, headers=headers)    html_content = response.text    match_dados = re.search(r"data\.addRows\(\[(.*?)\]\);", html_content, re.DOTALL)    colunas = re.findall(r"data\.addColumn\('.*?',\s*'(.*?)'\);", html_content)    if match_dados and colunas:        dados_js = match_dados.group(1)        dados_limpos = re.sub(r"\{v:\s*([^,]+),.*?\}", r"\1", dados_js)        lista_dados = ast.literal_eval(f"[{dados_limpos}]")        df = pd.DataFrame(lista_dados, columns=colunas)        return df    return pd.DataFrame()print("✅ Funções auxiliares carregadas.")

### 2.2 Extração dos 5 Datasets

In [32]:
# ==============================================================================# 2.2A — População Feminina Residente (IBGE via TabNet)# ==============================================================================def extrair_populacao():    url = "http://tabnet.datasus.gov.br/cgi/tabcgi.exe?ibge/cnv/popsvs2024br.def"    payload = "Linha=Unidade_da_Federa%E7%E3o&Coluna=Ano&Incremento=Popula%E7%E3o_residente&Arquivos=pop24.dbf&Arquivos=pop23.dbf&Arquivos=pop22.dbf&Arquivos=pop21.dbf&Arquivos=pop20.dbf&Arquivos=pop19.dbf&Arquivos=pop18.dbf&Arquivos=pop17.dbf&Arquivos=pop16.dbf&Arquivos=pop15.dbf&Arquivos=pop14.dbf&Arquivos=pop13.dbf&SRegi%E3o=TODAS_AS_CATEGORIAS__&pesqmes2=Digite+o+texto+e+ache+f%E1cil&SUnidade_da_Federa%E7%E3o=TODAS_AS_CATEGORIAS__&pesqmes3=Digite+o+texto+e+ache+f%E1cil&SMunic%EDpio=TODAS_AS_CATEGORIAS__&pesqmes4=Digite+o+texto+e+ache+f%E1cil&SCapital=TODAS_AS_CATEGORIAS__&pesqmes5=Digite+o+texto+e+ache+f%E1cil&SRegi%E3o_de_Sa%FAde_%28CIR%29=TODAS_AS_CATEGORIAS__&pesqmes6=Digite+o+texto+e+ache+f%E1cil&SMacrorregi%E3o_de_Sa%FAde=TODAS_AS_CATEGORIAS__&pesqmes7=Digite+o+texto+e+ache+f%E1cil&SMicrorregi%E3o_IBGE=TODAS_AS_CATEGORIAS__&pesqmes8=Digite+o+texto+e+ache+f%E1cil&SRegi%E3o_Metropolitana_-_RIDE=TODAS_AS_CATEGORIAS__&pesqmes9=Digite+o+texto+e+ache+f%E1cil&SMacrorregi%E3o_PNDR=TODAS_AS_CATEGORIAS__&SAmaz%F4nia_Legal=TODAS_AS_CATEGORIAS__&SSemi%E1rido=TODAS_AS_CATEGORIAS__&SFaixa_de_Fronteira=TODAS_AS_CATEGORIAS__&SZona_de_Fronteira=TODAS_AS_CATEGORIAS__&SMunic%EDpio_de_extrema_pobreza=TODAS_AS_CATEGORIAS__&SSexo=2&pesqmes16=Digite+o+texto+e+ache+f%E1cil&SFaixa_Et%E1ria_1=3&SFaixa_Et%E1ria_1=4&SFaixa_Et%E1ria_1=5&SFaixa_Et%E1ria_1=6&SFaixa_Et%E1ria_1=7&SFaixa_Et%E1ria_1=8&SFaixa_Et%E1ria_1=9&SFaixa_Et%E1ria_1=10&SFaixa_Et%E1ria_1=11&pesqmes17=Digite+o+texto+e+ache+f%E1cil&SFaixa_Et%E1ria_2=TODAS_AS_CATEGORIAS__&pesqmes18=Digite+o+texto+e+ache+f%E1cil&SIdade_simples=TODAS_AS_CATEGORIAS__&formato=table&mostre=Mostra"    headers = {        'Content-Type': 'application/x-www-form-urlencoded',        'User-Agent': 'Mozilla/5.0',        'Referer': url    }    return extrair_tabnet_html(url, payload, headers)df_populacao_feminina = carregar_ou_extrair('populacao_feminina', extrair_populacao)print(f"✅ População: {df_populacao_feminina.shape}")display(df_populacao_feminina.head(5))

In [33]:
# ==============================================================================# 2.2B — Óbitos por Neoplasia Maligna do Colo do Útero (SIM - CID-10 C53)# ==============================================================================def extrair_obitos():    url = "http://tabnet.datasus.gov.br/cgi/tabcgi.exe?sim/cnv/obt10uf.def"    payload = "Linha=Unidade_da_Federa%E7%E3o&Coluna=Ano_do_%D3bito&Incremento=%D3bitos_p%2FOcorr%EAnc&Arquivos=obtuf24.dbf&Arquivos=obtuf23.dbf&Arquivos=obtuf22.dbf&Arquivos=obtuf21.dbf&Arquivos=obtuf20.dbf&Arquivos=obtuf19.dbf&Arquivos=obtuf18.dbf&Arquivos=obtuf17.dbf&Arquivos=obtuf16.dbf&Arquivos=obtuf15.dbf&Arquivos=obtuf14.dbf&Arquivos=obtuf13.dbf&SRegi%E3o=TODAS_AS_CATEGORIAS__&pesqmes2=Digite+o+texto+e+ache+f%E1cil&SUnidade_da_Federa%E7%E3o=TODAS_AS_CATEGORIAS__&pesqmes3=&pesqmes4=Digite+o+texto+e+ache+f%E1cil&SGrupo_CID-10=TODAS_AS_CATEGORIAS__&pesqmes5=&SCategoria_CID-10=219&pesqmes6=Digite+o+texto+e+ache+f%E1cil&SCausa_-_CID-BR-10=TODAS_AS_CATEGORIAS__&SCausa_mal_definidas=TODAS_AS_CATEGORIAS__&pesqmes8=Digite+o+texto+e+ache+f%E1cil&SFaixa_Et%E1ria=TODAS_AS_CATEGORIAS__&pesqmes9=Digite+o+texto+e+ache+f%E1cil&SFaixa_Et%E1ria_OPS=TODAS_AS_CATEGORIAS__&pesqmes10=Digite+o+texto+e+ache+f%E1cil&SFaixa_Et%E1ria_det=TODAS_AS_CATEGORIAS__&SFx.Et%E1ria_Menor_1A=TODAS_AS_CATEGORIAS__&SSexo=TODAS_AS_CATEGORIAS__&SCor%2Fra%E7a=TODAS_AS_CATEGORIAS__&SEscolaridade=TODAS_AS_CATEGORIAS__&SEstado_civil=TODAS_AS_CATEGORIAS__&SLocal_ocorr%EAncia=TODAS_AS_CATEGORIAS__&formato=table&mostre=Mostra"    headers = {        'Content-Type': 'application/x-www-form-urlencoded',        'User-Agent': 'Mozilla/5.0',        'Referer': url    }    return extrair_tabnet_html(url, payload, headers)df_obitos = carregar_ou_extrair('obitos_neop_utero', extrair_obitos)print(f"✅ Óbitos: {df_obitos.shape}")display(df_obitos.head(5))

In [34]:
# ==============================================================================# 2.2C — Exames Citopatológicos (SISCAN) — Extração ano a ano# ==============================================================================def extrair_exames_cito():    session = requests.Session()    retries = Retry(total=5, backoff_factor=2, status_forcelist=[500,502,503,504], allowed_methods=["POST"])    session.mount('http://', HTTPAdapter(max_retries=retries))    url = "http://tabnet.datasus.gov.br/cgi/webtabx.exe?SISCAN/cito_colo_pacbr.def"    headers = {        'Content-Type': 'application/x-www-form-urlencoded',        'User-Agent': 'Mozilla/5.0',        'Referer': 'http://tabnet.datasus.gov.br/cgi/dhdat.exe?SISCAN/cito_colo_pacbr.def',        'cache-control': 'max-age=0'    }    session.headers.update(headers)    payload_base = "Linha=UF+de+residencia%7CCO_UF_RESIDENCIA%7C1%7Cterritorio%5Cbr_uf.cnv&Coluna=Ano+competencia%7CCO_ANO_LIBERACAO%7C1%7CCITO%5Cano.cnv&Incremento=Pacientes+distintos%7C%3Dcount%28distinct+co_paciente%29"    payload_filtros = "&pesqmes1=Digite+o+texto+e+ache+f%E1cil&SUF+de+residencia=TODAS_AS_CATEGORIAS__&pesqmes2=Digite+o+texto+e+ache+f%E1cil&SMunic.de+residencia=TODAS_AS_CATEGORIAS__&XSexo=TODAS_AS_CATEGORIAS__&XRa%E7a%2FCor=TODAS_AS_CATEGORIAS__&pesqmes6=Digite+o+texto+e+ache+f%E1cil&XFaixa+et%E1ria=Entre+10+a+14+anos%7C010-014%7C3&XFaixa+et%E1ria=Entre+15+a+19+anos%7C015-019%7C3&XFaixa+et%E1ria=Entre+20+a+24+anos%7C020-024%7C3&XFaixa+et%E1ria=Entre+25+a+29+anos%7C025-029%7C3&XFaixa+et%E1ria=Entre+30+a+34+anos%7C030-034%7C3&XFaixa+et%E1ria=Entre+35+a+39+anos%7C035-039%7C3&XFaixa+et%E1ria=Entre+40+a+44+anos%7C040-044%7C3&XFaixa+et%E1ria=Entre+45+a+49+anos%7C045-049%7C3&XFaixa+et%E1ria=Entre+50+a+54+anos%7C050-054%7C3&XFaixa+et%E1ria=Entre+55+a+59+anos%7C055-059%7C3&XFaixa+et%E1ria=Entre+60+a+64+anos%7C060-064%7C3&XFaixa+et%E1ria=Entre+65+a+69+anos%7C065-069%7C3&XFaixa+et%E1ria=Entre+70+a+74+anos%7C070-074%7C3&XFaixa+et%E1ria=Entre+75+a+79+anos%7C075-079%7C3&XFaixa+et%E1ria=Acima+de+79+anos%7C080-120%7C3&XEscolaridade=TODAS_AS_CATEGORIAS__&XCitologia+anterior=TODAS_AS_CATEGORIAS__&XAdequabilidade=TODAS_AS_CATEGORIAS__&pesqmes10=Digite+o+texto+e+ache+f%E1cil&XLaudo+Citopatol%F3gico=TODAS_AS_CATEGORIAS__&XPres.+Cel.+Endometri=TODAS_AS_CATEGORIAS__&XRepresent.+ZT=TODAS_AS_CATEGORIAS__&XMotivo+do+exame=TODAS_AS_CATEGORIAS__&XInspe%E7%E3o+do+colo=TODAS_AS_CATEGORIAS__&pesqmes15=Digite+o+texto+e+ache+f%E1cil&XAno+Resultado=TODAS_AS_CATEGORIAS__&nomedef=SISCAN%2Fcito_colo_pacbr.def&grafico="    lista_dfs = []    for ano in range(2013, 2025):        print(f"  Baixando {ano}...", end=" ")        parte_ano = f"&PAno+competencia={ano}%7C{ano}%7C4"        payload = payload_base + parte_ano + payload_filtros        try:            resp = session.post(url, data=payload, timeout=180)            resp.raise_for_status()            html = resp.text            match = re.search(r"data\.addRows\(\[(.*?)\]\);", html, re.DOTALL)            cols = re.findall(r"data\.addColumn\('.*?',\s*'(.*?)'\);", html)            if match and cols:                dados = re.sub(r"\{v:\s*([^,]+),.*?\}", r"\1", match.group(1))                df_t = pd.DataFrame(ast.literal_eval(f"[{dados}]"), columns=cols)                if 'Total' in df_t.iloc[:, 0].values:                    df_t = df_t[df_t.iloc[:, 0] != 'Total']                df_t.set_index(df_t.columns[0], inplace=True)                df_t.rename(columns={df_t.columns[0]: str(ano)}, inplace=True)                df_t = df_t[[str(ano)]]                lista_dfs.append(df_t)                print("✅")            else:                print("⚠️ Sem dados")        except Exception as e:            print(f"❌ {e}")        time.sleep(2)    if lista_dfs:        df_final = pd.concat(lista_dfs, axis=1).reset_index()        df_final.fillna(0, inplace=True)        split = df_final.iloc[:, 0].str.split(' ', n=1, expand=True)        df_final['id_uf'] = split[0]        df_final['uf_nome'] = split[1]        first_col = df_final.columns[0]        df_final.drop(columns=[first_col], inplace=True)        cols = ['id_uf', 'uf_nome'] + [c for c in df_final.columns if c not in ['id_uf', 'uf_nome']]        return df_final[cols].sort_values('id_uf')    return pd.DataFrame()df_exames = carregar_ou_extrair('exames_cito', extrair_exames_cito)print(f"✅ Exames Citopatológicos: {df_exames.shape}")display(df_exames.head(5))

In [35]:
# ==============================================================================# 2.2D — Imunizações HPV (SI-PNI)# ==============================================================================def extrair_imunizacoes():    url = "http://tabnet.datasus.gov.br/cgi/webtabx.exe?bd_pni/dpnibr.def"    payload = "Linha=Unidade+da+Federa%E7%E3o%7CFATO.CO_UF%7C1%7Cterritorio%5Cbr_uf.cnv&Coluna=Ano%7CCO_ANO%7C1%7CBD_pni%5CCNV%5CANO.CNV&Incremento=Doses_aplicadas%7CQT_DOSE&PAno=2022%7C2022%7C4&PAno=2021%7C2021%7C4&PAno=2020%7C2020%7C4&PAno=2019%7C2019%7C4&PAno=2018%7C2018%7C4&PAno=2017%7C2017%7C4&PAno=2016%7C2016%7C4&PAno=2015%7C2015%7C4&PAno=2014%7C2014%7C4&PAno=2013%7C2013%7C4&SRegi%E3o=TODAS_AS_CATEGORIAS__&pesqmes2=Digite+o+texto+e+ache+f%E1cil&SUnidade+da+Federa%E7%E3o=TODAS_AS_CATEGORIAS__&pesqmes3=Digite+o+texto+e+ache+f%E1cil&SMunic%EDpio=TODAS_AS_CATEGORIAS__&pesqmes4=Digite+o+texto+e+ache+f%E1cil&SCapital=TODAS_AS_CATEGORIAS__&pesqmes5=Digite+o+texto+e+ache+f%E1cil&SRegi%E3o+de+Sa%FAde+%28CIR%29=TODAS_AS_CATEGORIAS__&pesqmes6=Digite+o+texto+e+ache+f%E1cil&SMacrorregi%E3o+de+Sa%FAde=TODAS_AS_CATEGORIAS__&pesqmes7=Digite+o+texto+e+ache+f%E1cil&SMicrorregi%E3o+IBGE=TODAS_AS_CATEGORIAS__&pesqmes8=Digite+o+texto+e+ache+f%E1cil&SRegi%E3o+Metropolitana+-+RIDE=TODAS_AS_CATEGORIAS__&pesqmes9=Digite+o+texto+e+ache+f%E1cil&STerrit%F3rio+da+Cidadania=TODAS_AS_CATEGORIAS__&pesqmes10=Digite+o+texto+e+ache+f%E1cil&SMesorregi%E3o+PNDR=TODAS_AS_CATEGORIAS__&SAmaz%F4nia+Legal=TODAS_AS_CATEGORIAS__&SSemi%E1rido=TODAS_AS_CATEGORIAS__&SFaixa+de+Fronteira=TODAS_AS_CATEGORIAS__&SZona+de+Fronteira=TODAS_AS_CATEGORIAS__&SMunic%EDpio+de+extrema+pobreza=TODAS_AS_CATEGORIAS__&pesqmes16=&SImunobiol%F3gicos=HPV+Quadrivalente+-+Feminino%7C93%7C2&SImunobiol%F3gicos=HPV+Quadrivalente+-+Masculino%7C94%7C2&SImunobiol%F3gicos=HPV%7C84%7C2&pesqmes17=Digite+o+texto+e+ache+f%E1cil&SDose=TODAS_AS_CATEGORIAS__&pesqmes18=Digite+o+texto+e+ache+f%E1cil&SAno%2Fm%EAs=TODAS_AS_CATEGORIAS__&pesqmes20=Digite+o+texto+e+ache+f%E1cil&SFaixa_Et%E1ria=TODAS_AS_CATEGORIAS__&nomedef=bd_pni%2Fdpnibr.def&grafico="    headers = {        'Content-Type': 'application/x-www-form-urlencoded',        'User-Agent': 'Mozilla/5.0',        'Referer': url    }    df = extrair_tabnet_js(url, payload, headers)    if not df.empty:        df = df[df.iloc[:, 0] != 'Total'].copy()        split = df.iloc[:, 0].str.split(' ', n=1, expand=True)        df['id_uf'] = split[0]        df['uf_nome'] = split[1]        first_col = df.columns[0]        df.drop(columns=[first_col], inplace=True)        cols = ['id_uf', 'uf_nome'] + [c for c in df.columns if c not in ['id_uf', 'uf_nome']]        return df[cols]    return dfdf_imunizacoes = carregar_ou_extrair('imunizacoes_hpv', extrair_imunizacoes)print(f"✅ Imunizações HPV: {df_imunizacoes.shape}")display(df_imunizacoes.head(5))

In [36]:
# ==============================================================================# 2.2E — Diagnósticos Histopatológicos (SISCAN)# ==============================================================================def extrair_diagnosticos():    url = "http://tabnet.datasus.gov.br/cgi/webtabx.exe?siscan/histo_pacbr.def"    payload = "Linha=UF+de+residencia%7CCO_UF_RESIDENCIA%7C1%7Cterritorio%5Cbr_uf.cnv&Coluna=Ano+resultado%7CNU_ANO_RESULTADO%7C1%7CSISCAN%5Cano.cnv&Incremento=Pacientes+distintos%7C%3Dcount%28distinct+co_paciente%29&PAno+competencia=2025%7C2025%7C4&PAno+competencia=2024%7C2024%7C4&PAno+competencia=2023%7C2023%7C4&PAno+competencia=2022%7C2022%7C4&PAno+competencia=2021%7C2021%7C4&PAno+competencia=2020%7C2020%7C4&PAno+competencia=2019%7C2019%7C4&PAno+competencia=2018%7C2018%7C4&PAno+competencia=2017%7C2017%7C4&PAno+competencia=2016%7C2016%7C4&PAno+competencia=2015%7C2015%7C4&PAno+competencia=2014%7C2014%7C4&PAno+competencia=2013%7C2013%7C4&pesqmes1=Digite+o+texto+e+ache+f%E1cil&SUF+de+residencia=TODAS_AS_CATEGORIAS__&pesqmes2=Digite+o+texto+e+ache+f%E1cil&SMunic.de+residencia=TODAS_AS_CATEGORIAS__&pesqmes3=Digite+o+texto+e+ache+f%E1cil&SAno+resultado=TODAS_AS_CATEGORIAS__&SLaudo+histopatol%F3gico=Carcinoma+Epidermoide%7C01%7C2&SLaudo+histopatol%F3gico=Adenocarcinoma+invasor%7C02%7C2&SLaudo+histopatol%F3gico=Adenocarcinoma+in+situ%7C03%7C2&SLaudo+histopatol%F3gico=NIC+III+%2F+Carc.+in+situ%7C04%7C2&XRa%E7a%2FCor=TODAS_AS_CATEGORIAS__&XSexo=TODAS_AS_CATEGORIAS__&pesqmes8=Digite+o+texto+e+ache+f%E1cil&XFaixa+et%E1ria=TODAS_AS_CATEGORIAS__&XEscolaridade=TODAS_AS_CATEGORIAS__&XTipo+Encaminhamento=TODAS_AS_CATEGORIAS__&XTipo+de+procedimento+%28mat.+enviado%29=TODAS_AS_CATEGORIAS__&XAdequabilidade=Satisfat%F3rio%7C01%7C2&nomedef=siscan%2Fhisto_pacbr.def&grafico="    headers = {        'Content-Type': 'application/x-www-form-urlencoded',        'User-Agent': 'Mozilla/5.0',        'Referer': url    }    df = extrair_tabnet_js(url, payload, headers)    if not df.empty:        df = df[df.iloc[:, 0] != 'Total'].copy()        split = df.iloc[:, 0].str.split(' ', n=1, expand=True)        df['id_uf'] = split[0]        df['uf_nome'] = split[1]        first_col = df.columns[0]        df.drop(columns=[first_col], inplace=True)        cols = ['id_uf', 'uf_nome'] + [c for c in df.columns if c not in ['id_uf', 'uf_nome']]        return df[cols]    return dfdf_diagnosticos = carregar_ou_extrair('diagnosticos_histo', extrair_diagnosticos)print(f"✅ Diagnósticos Histopatológicos: {df_diagnosticos.shape}")display(df_diagnosticos.head(5))

## 3. Data Wrangling — Preparação e Engenharia de Atributos> Transformação dos 5 datasets extraídos (formato wide) em um **DataFrame Master** unificado> com KPIs calculados: taxa de mortalidade, cobertura de rastreio, positividade e cobertura vacinal.

In [37]:
# ==============================================================================# 3.1 — Função de Transformação Wide → Long# ==============================================================================def transformar_para_long(df, nome_valor, colunas_fixas=['id_uf', 'uf_nome']):    """Transforma DataFrame wide (anos em colunas) para formato long."""    if 'id_uf' in df.columns:        df = df[pd.to_numeric(df['id_uf'], errors='coerce').notnull()].copy()    colunas_anos = [c for c in df.columns if c not in colunas_fixas and str(c).strip().isdigit()]    df_long = pd.melt(        df, id_vars=colunas_fixas, value_vars=colunas_anos,        var_name='ano', value_name=nome_valor    )    df_long['ano'] = pd.to_numeric(df_long['ano'], errors='coerce')    df_long = df_long.dropna(subset=['ano'])    df_long['ano'] = df_long['ano'].astype(int)    if df_long[nome_valor].dtype == 'object':        df_long[nome_valor] = (df_long[nome_valor].astype(str)                               .str.replace('.', '', regex=False)                               .str.replace(',', '.', regex=False))    df_long[nome_valor] = pd.to_numeric(df_long[nome_valor], errors='coerce').fillna(0)    return df_longprint("✅ Função de transformação carregada.")

In [38]:
# ==============================================================================# 3.2 — Transformação e Integração dos 5 Datasets# ==============================================================================# Checar se df_master já está em cachecache_master = os.path.join(DATA_DIR, 'df_master.csv')if os.path.exists(cache_master):    print("📂 Carregando df_master do cache...")    df_master = pd.read_csv(cache_master)else:    print("🔧 Construindo df_master a partir dos datasets brutos...")    # A. Transformar cada dataset para formato long    df_pop_long = transformar_para_long(df_populacao_feminina, 'populacao')    df_obitos_long = transformar_para_long(df_obitos, 'obitos')    df_exames_long = transformar_para_long(df_exames, 'exames_realizados')    df_diag_long = transformar_para_long(df_diagnosticos, 'diagnosticos_positivos')    df_imun_long = transformar_para_long(df_imunizacoes, 'doses_aplicadas')    # B. Merge progressivo    df_master = df_pop_long.merge(df_obitos_long, on=['id_uf', 'uf_nome', 'ano'], how='left')    df_master = df_master.merge(df_exames_long, on=['id_uf', 'uf_nome', 'ano'], how='left')    df_master = df_master.merge(df_diag_long, on=['id_uf', 'uf_nome', 'ano'], how='left')    df_master = df_master.merge(df_imun_long, on=['id_uf', 'uf_nome', 'ano'], how='left')    df_master = df_master.fillna(0)    # Limpar linhas sem UF válida    df_master = df_master[pd.to_numeric(df_master['id_uf'], errors='coerce').notnull()]    df_master['id_uf'] = df_master['id_uf'].astype(int)    # C. Feature Engineering — KPIs    df_master['taxa_mortalidade'] = np.where(        df_master['populacao'] > 0,        (df_master['obitos'] / df_master['populacao']) * 100000, 0    )    df_master['razao_exames_pop'] = np.where(        df_master['populacao'] > 0,        (df_master['exames_realizados'] / df_master['populacao']) * 100, 0    )    df_master['taxa_positividade'] = np.where(        df_master['exames_realizados'] > 0,        (df_master['diagnosticos_positivos'] / df_master['exames_realizados']) * 100, 0    )    df_master['cobertura_vacinal_100k'] = np.where(        df_master['populacao'] > 0,        (df_master['doses_aplicadas'] / df_master['populacao']) * 100000, 0    )    # D. Classificação por Região    def classificar_regiao(uf):        mapa = {            'Norte': ['Acre','Amazonas','Roraima','Rondônia','Pará','Amapá','Tocantins'],            'Nordeste': ['Maranhão','Piauí','Ceará','Rio Grande do Norte','Paraíba','Pernambuco','Alagoas','Sergipe','Bahia'],            'Sudeste': ['Minas Gerais','Espírito Santo','Rio de Janeiro','São Paulo'],            'Sul': ['Paraná','Santa Catarina','Rio Grande do Sul'],            'Centro-Oeste': ['Mato Grosso do Sul','Mato Grosso','Goiás','Distrito Federal']        }        for regiao, estados in mapa.items():            if uf in estados:                return regiao        return 'Outros'    df_master['regiao'] = df_master['uf_nome'].apply(classificar_regiao)    # E. Siglas dos estados (para gráficos compactos)    SIGLAS = {        'Rondônia':'RO','Acre':'AC','Amazonas':'AM','Roraima':'RR','Pará':'PA',        'Amapá':'AP','Tocantins':'TO','Maranhão':'MA','Piauí':'PI','Ceará':'CE',        'Rio Grande do Norte':'RN','Paraíba':'PB','Pernambuco':'PE','Alagoas':'AL',        'Sergipe':'SE','Bahia':'BA','Minas Gerais':'MG','Espírito Santo':'ES',        'Rio de Janeiro':'RJ','São Paulo':'SP','Paraná':'PR','Santa Catarina':'SC',        'Rio Grande do Sul':'RS','Mato Grosso do Sul':'MS','Mato Grosso':'MT',        'Goiás':'GO','Distrito Federal':'DF'    }    df_master['sigla_uf'] = df_master['uf_nome'].map(SIGLAS)    # F. Código IBGE para GeoJSON (2 dígitos)    df_master['cod_uf_geo'] = df_master['id_uf'].astype(str).str.zfill(2)    # Salvar cache    salvar_cache(df_master, 'df_master')print(f"\n✅ df_master pronto: {df_master.shape[0]} linhas × {df_master.shape[1]} colunas")print(f"📊 Período: {df_master['ano'].min()} – {df_master['ano'].max()}")print(f"🗺️ Estados: {df_master['uf_nome'].nunique()}")display(df_master.head(10))

In [39]:
# ==============================================================================# 3.3 — Resumo Estatístico Descritivo# ==============================================================================print("📋 Estatísticas Descritivas dos KPIs:")print("="*60)display(df_master[['taxa_mortalidade', 'razao_exames_pop', 'taxa_positividade',                   'cobertura_vacinal_100k']].describe().round(2))print("\n📋 Valores por Região (médias):")display(df_master.groupby('regiao')[['taxa_mortalidade', 'razao_exames_pop']].mean().round(2))

## 4. Análise Exploratória — Visualizações Interativas> Todos os gráficos abaixo são **interativos**: hover para detalhes, zoom, pan, e filtros.> Construídos com Plotly para máxima interatividade e qualidade visual.

### 4.1 🗺️ Mapa Coroplético — Taxa de Mortalidade por Estado

In [40]:
# ==============================================================================# 4.1 — Mapa Coroplético Animado: Mortalidade por UF ao Longo dos Anos# ==============================================================================# Baixar GeoJSON dos estados brasileirosimport urllib.requestgeojson_path = os.path.join(DATA_DIR, 'brazil_states.json')if not os.path.exists(geojson_path):    print("📥 Baixando GeoJSON dos estados brasileiros...")    url_geo = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"    urllib.request.urlretrieve(url_geo, geojson_path)    print("✅ GeoJSON salvo!")with open(geojson_path) as f:    geojson_br = json.load(f)# Mapear siglas para o GeoJSONfor feat in geojson_br['features']:    feat['id'] = feat['properties'].get('sigla', feat['properties'].get('name', ''))fig_mapa = px.choropleth(    df_master,    geojson=geojson_br,    locations='sigla_uf',    color='taxa_mortalidade',    animation_frame='ano',    color_continuous_scale='YlOrRd',    range_color=[0, df_master['taxa_mortalidade'].quantile(0.95)],    scope='south america',    hover_name='uf_nome',    hover_data={        'taxa_mortalidade': ':.2f',        'obitos': ':,.0f',        'populacao': ':,.0f',        'sigla_uf': False    },    labels={'taxa_mortalidade': 'Mortalidade (por 100k)', 'ano': 'Ano'},    title='🗺️ Taxa de Mortalidade por Câncer de Colo do Útero — Brasil (por 100 mil mulheres)')fig_mapa.update_geos(    fitbounds="locations",    visible=False,    bgcolor='rgba(0,0,0,0)')fig_mapa.update_layout(    height=650,    margin=dict(l=0, r=0, t=60, b=0),    paper_bgcolor='#1A1A2E',    font_color='#E0E0E0',    title_font_size=18)fig_mapa.show()

### 4.2 📈 Série Temporal — Evolução da Mortalidade Nacional

In [41]:
# ==============================================================================# 4.2 — Série Temporal Interativa: Mortalidade Brasil# ==============================================================================df_brasil = df_master.groupby('ano').agg(    obitos_total=('obitos', 'sum'),    pop_total=('populacao', 'sum')).reset_index()df_brasil['taxa_br'] = (df_brasil['obitos_total'] / df_brasil['pop_total']) * 100000fig_ts = go.Figure()# Linha principalfig_ts.add_trace(go.Scatter(    x=df_brasil['ano'], y=df_brasil['taxa_br'],    mode='lines+markers',    name='Taxa Nacional',    line=dict(color='#E63946', width=3),    marker=dict(size=8, symbol='circle'),    hovertemplate='<b>%{x}</b><br>Taxa: %{y:.2f}/100k<extra></extra>'))# Média móveldf_brasil['mm3'] = df_brasil['taxa_br'].rolling(3, center=True).mean()fig_ts.add_trace(go.Scatter(    x=df_brasil['ano'], y=df_brasil['mm3'],    mode='lines',    name='Média Móvel (3 anos)',    line=dict(color='#457B9D', width=2, dash='dash'),    hovertemplate='<b>%{x}</b><br>MM3: %{y:.2f}/100k<extra></extra>'))# Destaque COVIDfig_ts.add_vrect(x0=2019.5, x1=2021.5, fillcolor="#F4A261", opacity=0.15,                 line_width=0, annotation_text="Pandemia COVID-19",                 annotation_position="top left", annotation_font_color="#F4A261")fig_ts.update_layout(    title='📈 Evolução da Taxa de Mortalidade por Câncer de Colo do Útero — Brasil',    xaxis_title='Ano', yaxis_title='Óbitos por 100 mil mulheres',    height=500, hovermode='x unified',    paper_bgcolor='#1A1A2E', plot_bgcolor='#16213E',    font_color='#E0E0E0', title_font_size=16,    legend=dict(orientation='h', yanchor='bottom', y=1.02))fig_ts.show()

### 4.3 🏆 Bar Chart Race — Top 10 Estados com Maior Mortalidade

In [42]:
# ==============================================================================# 4.3 — Ranking Animado: Top 10 UFs por Mortalidade# ==============================================================================df_rank = df_master.sort_values(['ano', 'taxa_mortalidade'], ascending=[True, False])df_top10 = df_rank.groupby('ano').head(10).reset_index(drop=True)fig_race = px.bar(    df_top10,    x='taxa_mortalidade', y='sigla_uf',    color='regiao',    animation_frame='ano',    orientation='h',    color_discrete_map=PALETA_REGIOES,    hover_name='uf_nome',    hover_data={'taxa_mortalidade': ':.2f', 'obitos': ':,.0f', 'sigla_uf': False},    labels={'taxa_mortalidade': 'Mortalidade (por 100k)', 'sigla_uf': 'Estado'},    title='🏆 Top 10 Estados com Maior Mortalidade por Ano')fig_race.update_layout(    height=550, yaxis={'categoryorder': 'total ascending'},    paper_bgcolor='#1A1A2E', plot_bgcolor='#16213E',    font_color='#E0E0E0', title_font_size=16,    legend_title='Região')fig_race.show()

### 4.4 🔬 Scatter — Cobertura vs Mortalidade (Bubble Chart)

In [43]:
# ==============================================================================# 4.4 — Scatter Interativo: Cobertura vs Mortalidade# ==============================================================================# Agregar médias por UF (excluindo anos com dados incompletos)df_scatter = df_master[df_master['ano'].between(2014, 2022)].groupby(    ['uf_nome', 'sigla_uf', 'regiao']).agg(    media_cobertura=('razao_exames_pop', 'mean'),    media_mortalidade=('taxa_mortalidade', 'mean'),    pop_media=('populacao', 'mean'),    total_obitos=('obitos', 'sum')).reset_index()fig_scatter = px.scatter(    df_scatter,    x='media_cobertura', y='media_mortalidade',    size='pop_media', color='regiao',    hover_name='uf_nome',    size_max=50,    color_discrete_map=PALETA_REGIOES,    trendline='ols',    hover_data={        'media_cobertura': ':.2f',        'media_mortalidade': ':.2f',        'pop_media': ':,.0f',        'total_obitos': ':,.0f'    },    labels={        'media_cobertura': 'Cobertura de Exames (%)',        'media_mortalidade': 'Mortalidade (por 100k)',        'pop_media': 'População Feminina'    },    title='🔬 Cobertura de Rastreamento vs. Mortalidade (Média 2014-2022)')# Adicionar rótulos dos estadosfor _, row in df_scatter.iterrows():    fig_scatter.add_annotation(        x=row['media_cobertura'], y=row['media_mortalidade'],        text=row['sigla_uf'], showarrow=False,        font=dict(size=9, color='#E0E0E0'), yshift=12    )fig_scatter.update_layout(    height=600,    paper_bgcolor='#1A1A2E', plot_bgcolor='#16213E',    font_color='#E0E0E0', title_font_size=16,    legend_title='Região')fig_scatter.show()

### 4.5 🌡️ Heatmap Temporal — Mortalidade por UF × Ano

In [44]:
# ==============================================================================# 4.5 — Heatmap Temporal: UF × Ano (Mortalidade)# ==============================================================================pivot_mort = df_master.pivot_table(    values='taxa_mortalidade', index='sigla_uf', columns='ano').fillna(0)# Ordenar por médiapivot_mort = pivot_mort.loc[pivot_mort.mean(axis=1).sort_values(ascending=False).index]fig_heat = px.imshow(    pivot_mort,    color_continuous_scale='YlOrRd',    aspect='auto',    labels={'x': 'Ano', 'y': 'Estado', 'color': 'Mortalidade/100k'},    title='🌡️ Heatmap: Taxa de Mortalidade por Estado e Ano')fig_heat.update_layout(    height=750,    paper_bgcolor='#1A1A2E',    font_color='#E0E0E0', title_font_size=16,    xaxis_title='Ano', yaxis_title='Estado (UF)')fig_heat.show()

### 4.6 🌐 Sunburst — Distribuição de Óbitos por Região e Estado

In [45]:
# ==============================================================================# 4.6 — Sunburst: Hierarquia Região → Estado → Óbitos# ==============================================================================ultimo_ano = df_master['ano'].max()df_sun = df_master[df_master['ano'] == ultimo_ano][['regiao', 'uf_nome', 'obitos', 'taxa_mortalidade']].copy()df_sun = df_sun[df_sun['regiao'] != 'Outros']fig_sun = px.sunburst(    df_sun,    path=['regiao', 'uf_nome'],    values='obitos',    color='taxa_mortalidade',    color_continuous_scale='YlOrRd',    hover_data={'taxa_mortalidade': ':.2f', 'obitos': ':,.0f'},    title=f'🌐 Distribuição de Óbitos por Região e Estado ({ultimo_ano})')fig_sun.update_layout(    height=600,    paper_bgcolor='#1A1A2E',    font=dict(color='#E0E0E0', size=12),    title_font_size=16)fig_sun.show()

### 4.7 🌳 Treemap — Proporção de Óbitos por Região/Estado

In [46]:
# ==============================================================================# 4.7 — Treemap: Distribuição Proporcional de Óbitos# ==============================================================================df_tree = df_master.groupby(['regiao', 'uf_nome', 'sigla_uf']).agg(    total_obitos=('obitos', 'sum'),    media_mort=('taxa_mortalidade', 'mean')).reset_index()df_tree = df_tree[df_tree['regiao'] != 'Outros']fig_tree = px.treemap(    df_tree,    path=[px.Constant("Brasil"), 'regiao', 'uf_nome'],    values='total_obitos',    color='media_mort',    color_continuous_scale='YlOrRd',    hover_data={'total_obitos': ':,.0f', 'media_mort': ':.2f'},    title='🌳 Treemap: Distribuição de Óbitos Acumulados por Região e Estado (2013-2024)')fig_tree.update_layout(    height=600, paper_bgcolor='#1A1A2E',    font=dict(color='#E0E0E0'), title_font_size=16)fig_tree.show()

### 4.8 📊 Boxplot Interativo — Mortalidade e Cobertura por Região

In [47]:
# ==============================================================================# 4.8 — Boxplot Interativo por Região# ==============================================================================fig_box = make_subplots(rows=1, cols=2,    subplot_titles=['Taxa de Mortalidade (por 100k)', 'Cobertura de Exames (%)'],    horizontal_spacing=0.12)for regiao, cor in PALETA_REGIOES.items():    df_r = df_master[df_master['regiao'] == regiao]    fig_box.add_trace(go.Box(        y=df_r['taxa_mortalidade'], name=regiao,        marker_color=cor, boxmean='sd',        hoverinfo='y+name', showlegend=True    ), row=1, col=1)    fig_box.add_trace(go.Box(        y=df_r['razao_exames_pop'], name=regiao,        marker_color=cor, boxmean='sd',        hoverinfo='y+name', showlegend=False    ), row=1, col=2)fig_box.update_layout(    title='📊 Distribuição de Mortalidade e Cobertura por Região',    height=500, paper_bgcolor='#1A1A2E', plot_bgcolor='#16213E',    font_color='#E0E0E0', title_font_size=16,    legend=dict(orientation='h', yanchor='bottom', y=1.05))fig_box.show()

### 4.9 📊 Painel Multi-Métrico — Cobertura × Positividade × Mortalidade

In [48]:
# ==============================================================================# 4.9 — Dashboard Multi-Métrico (3 Painéis)# ==============================================================================df_br = df_master.groupby('ano').agg(    exames=('exames_realizados', 'sum'),    pop=('populacao', 'sum'),    diag=('diagnosticos_positivos', 'sum'),    obt=('obitos', 'sum')).reset_index()df_br['cobertura'] = (df_br['exames'] / df_br['pop']) * 100df_br['positividade'] = (df_br['diag'] / df_br['exames']) * 100df_br['mortalidade'] = (df_br['obt'] / df_br['pop']) * 100000fig_multi = make_subplots(    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,    subplot_titles=['Cobertura de Rastreamento (%)', 'Taxa de Positividade (%)', 'Mortalidade (por 100k)'])fig_multi.add_trace(go.Scatter(    x=df_br['ano'], y=df_br['cobertura'], mode='lines+markers',    line=dict(color='#2A9D8F', width=3), marker=dict(size=7),    name='Cobertura', hovertemplate='%{y:.2f}%<extra></extra>'), row=1, col=1)fig_multi.add_trace(go.Scatter(    x=df_br['ano'], y=df_br['positividade'], mode='lines+markers',    line=dict(color='#F4A261', width=3), marker=dict(size=7),    name='Positividade', hovertemplate='%{y:.4f}%<extra></extra>'), row=2, col=1)fig_multi.add_trace(go.Scatter(    x=df_br['ano'], y=df_br['mortalidade'], mode='lines+markers',    line=dict(color='#E63946', width=3), marker=dict(size=7),    name='Mortalidade', hovertemplate='%{y:.2f}/100k<extra></extra>'), row=3, col=1)fig_multi.update_layout(    title='📊 Panorama Nacional: Rastreamento, Positividade e Mortalidade',    height=700, paper_bgcolor='#1A1A2E', plot_bgcolor='#16213E',    font_color='#E0E0E0', title_font_size=16, showlegend=False,    hovermode='x unified')fig_multi.show()

### 4.10 🔻 Funil de Rastreamento — Pipeline de Saúde Pública

In [49]:
# ==============================================================================# 4.10 — Funnel: Pipeline do Rastreamento# ==============================================================================ultimo = df_master['ano'].max()df_fun = df_master[df_master['ano'] == ultimo]etapas = ['População Alvo', 'Examinadas (Papanicolau)', 'Diagnósticos Positivos', 'Óbitos']valores = [    df_fun['populacao'].sum(),    df_fun['exames_realizados'].sum(),    df_fun['diagnosticos_positivos'].sum(),    df_fun['obitos'].sum()]fig_funnel = go.Figure(go.Funnel(    y=etapas, x=valores,    textinfo='value+percent initial',    textposition='inside',    marker=dict(color=['#457B9D', '#2A9D8F', '#F4A261', '#E63946']),    connector=dict(line=dict(color='#E0E0E0', width=1))))fig_funnel.update_layout(    title=f'🔻 Funil do Rastreamento do Câncer de Colo do Útero ({ultimo})',    height=450, paper_bgcolor='#1A1A2E', plot_bgcolor='#16213E',    font_color='#E0E0E0', title_font_size=16)fig_funnel.show()

### 4.11 📈 Evolução Regional — Small Multiples por Região

In [50]:
# ==============================================================================# 4.11 — Small Multiples: Mortalidade por Região ao Longo do Tempo# ==============================================================================df_reg = df_master[df_master['regiao'] != 'Outros'].groupby(    ['ano', 'regiao']).agg(obt=('obitos','sum'), pop=('populacao','sum')).reset_index()df_reg['taxa'] = (df_reg['obt'] / df_reg['pop']) * 100000fig_sm = px.line(    df_reg, x='ano', y='taxa', color='regiao',    facet_col='regiao', facet_col_wrap=3,    color_discrete_map=PALETA_REGIOES,    markers=True,    labels={'taxa': 'Mortalidade/100k', 'ano': 'Ano'},    title='📈 Evolução da Mortalidade por Região')fig_sm.update_layout(    height=500, paper_bgcolor='#1A1A2E', plot_bgcolor='#16213E',    font_color='#E0E0E0', title_font_size=16, showlegend=False)fig_sm.for_each_annotation(lambda a: a.update(text=a.text.split("=")[1]))fig_sm.show()

### 4.12 🕸️ Radar Chart — Perfil Multidimensional por Região

In [51]:
# ==============================================================================# 4.12 — Radar Chart: Perfil das Regiões# ==============================================================================df_radar = df_master[df_master['regiao'] != 'Outros'].groupby('regiao').agg(    mortalidade=('taxa_mortalidade', 'mean'),    cobertura=('razao_exames_pop', 'mean'),    positividade=('taxa_positividade', 'mean'),    vacinal=('cobertura_vacinal_100k', 'mean')).reset_index()# Normalizar (0-1) para comparabilidadefor col in ['mortalidade', 'cobertura', 'positividade', 'vacinal']:    mx = df_radar[col].max()    if mx > 0:        df_radar[col + '_norm'] = df_radar[col] / mx    else:        df_radar[col + '_norm'] = 0categorias = ['Mortalidade', 'Cobertura Exames', 'Positividade', 'Cobertura Vacinal']fig_radar = go.Figure()for _, row in df_radar.iterrows():    vals = [row['mortalidade_norm'], row['cobertura_norm'],            row['positividade_norm'], row['vacinal_norm']]    vals.append(vals[0])  # fechar o radar    fig_radar.add_trace(go.Scatterpolar(        r=vals, theta=categorias + [categorias[0]],        fill='toself', name=row['regiao'],        line_color=PALETA_REGIOES.get(row['regiao'], '#888'),        opacity=0.7    ))fig_radar.update_layout(    polar=dict(        bgcolor='#16213E',        radialaxis=dict(visible=True, range=[0, 1.1], gridcolor='#333'),        angularaxis=dict(gridcolor='#333')    ),    title='🕸️ Perfil Multidimensional por Região (Normalizado)',    height=550, paper_bgcolor='#1A1A2E',    font_color='#E0E0E0', title_font_size=16,    legend=dict(orientation='h', yanchor='bottom', y=-0.15))fig_radar.show()

## 5. Análise Estatística Avançada> Testes de correlação (Pearson, Spearman, Kendall), ANOVA/Kruskal-Wallis,> Regressão Linear, PCA interativo 3D, Clusterização K-Means com Silhouette.

### 5.1 📐 Análise de Correlação — Pearson, Spearman e Kendall

In [52]:
# ==============================================================================# 5.1 — Correlações Multimétodo com Heatmap Interativo# ==============================================================================df_corr_periodo = df_master[df_master['ano'] <= 2022].copy()df_agg = df_corr_periodo.groupby('uf_nome').agg(    media_mortalidade=('taxa_mortalidade', 'mean'),    media_cobertura=('razao_exames_pop', 'mean'),    media_positividade=('taxa_positividade', 'mean'),    media_cobertura_vacinal=('cobertura_vacinal_100k', 'mean'),    total_obitos=('obitos', 'sum'),    total_diagnosticos=('diagnosticos_positivos', 'sum'),    total_doses_hpv=('doses_aplicadas', 'sum'),    total_exames=('exames_realizados', 'sum')).reset_index()# Matrizes de correlaçãovars_corr = ['media_mortalidade', 'media_cobertura', 'media_positividade', 'media_cobertura_vacinal']labels_pt = ['Mortalidade', 'Cobertura Exames', 'Positividade', 'Cobertura Vacinal']corr_pearson = df_agg[vars_corr].corr(method='pearson')corr_spearman = df_agg[vars_corr].corr(method='spearman')fig_corr = make_subplots(rows=1, cols=2,    subplot_titles=['Correlação Pearson', 'Correlação Spearman'],    horizontal_spacing=0.15)fig_corr.add_trace(go.Heatmap(    z=corr_pearson.values, x=labels_pt, y=labels_pt,    colorscale='RdBu_r', zmin=-1, zmax=1, showscale=False,    text=corr_pearson.values.round(3), texttemplate='%{text}',    textfont=dict(size=11)), row=1, col=1)fig_corr.add_trace(go.Heatmap(    z=corr_spearman.values, x=labels_pt, y=labels_pt,    colorscale='RdBu_r', zmin=-1, zmax=1, showscale=True,    text=corr_spearman.values.round(3), texttemplate='%{text}',    textfont=dict(size=11),    colorbar=dict(title='ρ', x=1.02)), row=1, col=2)fig_corr.update_layout(    title='📐 Matriz de Correlação: Pearson vs Spearman (Médias por UF)',    height=450, paper_bgcolor='#1A1A2E',    font_color='#E0E0E0', title_font_size=16)fig_corr.show()# Imprimir correlações-chaveprint("\n📊 Correlações-Chave (Spearman):")rho, p = stats.spearmanr(df_agg['media_cobertura'], df_agg['media_mortalidade'])print(f"  Cobertura vs Mortalidade: ρ={rho:.4f}, p={p:.4f} {'✅ Significativo' if p<0.05 else '⚠️ Não significativo'}")rho2, p2 = stats.spearmanr(df_agg['total_exames'], df_agg['total_diagnosticos'])print(f"  Exames vs Diagnósticos:   ρ={rho2:.4f}, p={p2:.4f}")

### 5.2 📊 ANOVA / Kruskal-Wallis — Diferenças Regionais

In [53]:
# ==============================================================================# 5.2 — ANOVA e Kruskal-Wallis por Região# ==============================================================================regioes = df_master[df_master['regiao'] != 'Outros']['regiao'].unique()grupos_mort = [df_master[df_master['regiao']==r]['taxa_mortalidade'].values for r in regioes]grupos_cob = [df_master[df_master['regiao']==r]['razao_exames_pop'].values for r in regioes]# ANOVAf_mort, p_mort = stats.f_oneway(*grupos_mort)f_cob, p_cob = stats.f_oneway(*grupos_cob)# Kruskal-Wallis (não-paramétrico)h_mort, pk_mort = stats.kruskal(*grupos_mort)h_cob, pk_cob = stats.kruskal(*grupos_cob)print("="*60)print("TESTE DE DIFERENÇAS ENTRE REGIÕES")print("="*60)print(f"\nMortalidade:")print(f"  ANOVA:          F={f_mort:.3f}, p={p_mort:.6f} {'✅' if p_mort<0.05 else '⚠️'}")print(f"  Kruskal-Wallis: H={h_mort:.3f}, p={pk_mort:.6f} {'✅' if pk_mort<0.05 else '⚠️'}")print(f"\nCobertura:")print(f"  ANOVA:          F={f_cob:.3f}, p={p_cob:.6f} {'✅' if p_cob<0.05 else '⚠️'}")print(f"  Kruskal-Wallis: H={h_cob:.3f}, p={pk_cob:.6f} {'✅' if pk_cob<0.05 else '⚠️'}")

### 5.3 📉 Regressão Linear — Cobertura → Mortalidade

In [54]:
# ==============================================================================# 5.3 — Regressão Linear com Gráfico de Resíduos# ==============================================================================X = df_agg[['media_cobertura']].valuesy = df_agg['media_mortalidade'].valuesmodel = LinearRegression().fit(X, y)y_pred = model.predict(X)residuos = y - y_predprint(f"Coeficiente Angular (β₁): {model.coef_[0]:.4f}")print(f"Intercepto (β₀): {model.intercept_:.4f}")print(f"R²: {model.score(X, y):.4f}")fig_reg = make_subplots(rows=1, cols=2,    subplot_titles=['Regressão: Cobertura → Mortalidade', 'Resíduos'])fig_reg.add_trace(go.Scatter(    x=df_agg['media_cobertura'], y=df_agg['media_mortalidade'],    mode='markers+text', text=df_agg['uf_nome'].str[:3],    textposition='top center', textfont=dict(size=8),    marker=dict(color='#457B9D', size=8), name='UFs',    hovertext=df_agg['uf_nome']), row=1, col=1)x_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)fig_reg.add_trace(go.Scatter(    x=x_range.flatten(), y=model.predict(x_range),    mode='lines', line=dict(color='#E63946', width=2, dash='dash'),    name=f'Regressão (R²={model.score(X,y):.3f})'), row=1, col=1)fig_reg.add_trace(go.Scatter(    x=y_pred, y=residuos, mode='markers',    marker=dict(color='#F4A261', size=7), name='Resíduos'), row=1, col=2)fig_reg.add_hline(y=0, line_dash='dash', line_color='white', row=1, col=2)fig_reg.update_layout(    height=450, paper_bgcolor='#1A1A2E', plot_bgcolor='#16213E',    font_color='#E0E0E0', title_font_size=14, showlegend=True)fig_reg.show()

### 5.4 🧬 PCA 3D — Redução de Dimensionalidade

In [55]:
# ==============================================================================# 5.4 — PCA Interativo 3D# ==============================================================================features = ['media_mortalidade', 'media_cobertura', 'media_positividade', 'media_cobertura_vacinal']X_pca = StandardScaler().fit_transform(df_agg[features])pca = PCA(n_components=3)coords = pca.fit_transform(X_pca)df_pca = pd.DataFrame(coords, columns=['PC1', 'PC2', 'PC3'])df_pca['uf_nome'] = df_agg['uf_nome'].values# Adicionar regiãosigla_regiao = df_master.drop_duplicates('uf_nome')[['uf_nome', 'regiao']]df_pca = df_pca.merge(sigla_regiao, on='uf_nome', how='left')print(f"Variância explicada: PC1={pca.explained_variance_ratio_[0]:.1%}, "      f"PC2={pca.explained_variance_ratio_[1]:.1%}, PC3={pca.explained_variance_ratio_[2]:.1%}")print(f"Total: {sum(pca.explained_variance_ratio_):.1%}")fig_pca = px.scatter_3d(    df_pca, x='PC1', y='PC2', z='PC3',    color='regiao', hover_name='uf_nome',    color_discrete_map=PALETA_REGIOES,    title='🧬 PCA 3D — Perfil dos Estados Brasileiros',    labels={'PC1': f'PC1 ({pca.explained_variance_ratio_[0]:.0%})',            'PC2': f'PC2 ({pca.explained_variance_ratio_[1]:.0%})',            'PC3': f'PC3 ({pca.explained_variance_ratio_[2]:.0%})'})fig_pca.update_layout(    height=600, paper_bgcolor='#1A1A2E',    font_color='#E0E0E0', title_font_size=16,    scene=dict(bgcolor='#16213E'))fig_pca.show()

### 5.5 🎯 Clusterização K-Means com Silhouette Score

In [56]:
# ==============================================================================# 5.5 — KMeans com Elbow Method e Silhouette# ==============================================================================X_km = StandardScaler().fit_transform(df_agg[features])# Elbow + Silhouetteinertias = []silhouettes = []K_range = range(2, 8)for k in K_range:    km = KMeans(n_clusters=k, random_state=42, n_init=10)    labels = km.fit_predict(X_km)    inertias.append(km.inertia_)    silhouettes.append(silhouette_score(X_km, labels))fig_elbow = make_subplots(rows=1, cols=2,    subplot_titles=['Elbow Method (Inércia)', 'Silhouette Score'])fig_elbow.add_trace(go.Scatter(    x=list(K_range), y=inertias, mode='lines+markers',    line=dict(color='#457B9D', width=2), marker=dict(size=8)), row=1, col=1)fig_elbow.add_trace(go.Scatter(    x=list(K_range), y=silhouettes, mode='lines+markers',    line=dict(color='#2A9D8F', width=2), marker=dict(size=8)), row=1, col=2)fig_elbow.update_layout(    height=350, paper_bgcolor='#1A1A2E', plot_bgcolor='#16213E',    font_color='#E0E0E0', showlegend=False)fig_elbow.show()# KMeans com k=3best_k = 3km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)df_agg['cluster'] = km_final.fit_predict(X_km)df_agg['cluster_label'] = df_agg['cluster'].map({    0: 'Cluster A', 1: 'Cluster B', 2: 'Cluster C'})fig_km = px.scatter(    df_agg, x='media_cobertura', y='media_mortalidade',    color='cluster_label', hover_name='uf_nome',    size='total_obitos', size_max=30,    color_discrete_sequence=['#E63946', '#2A9D8F', '#457B9D'],    title=f'🎯 Clusterização K-Means (k={best_k}) — Perfis de UFs',    labels={'media_cobertura': 'Cobertura (%)', 'media_mortalidade': 'Mortalidade/100k'})# Adicionar rótulosfor _, row in df_agg.iterrows():    fig_km.add_annotation(x=row['media_cobertura'], y=row['media_mortalidade'],                          text=row['uf_nome'][:3], showarrow=False,                          font=dict(size=8, color='#E0E0E0'), yshift=10)fig_km.update_layout(    height=550, paper_bgcolor='#1A1A2E', plot_bgcolor='#16213E',    font_color='#E0E0E0', title_font_size=16)fig_km.show()# Estatísticas por clusterprint("\n📊 Perfil dos Clusters:")display(df_agg.groupby('cluster_label')[features].mean().round(2))

## 6. Dashboard Consolidado — KPI Cards> Painel final com indicadores-chave do último ano disponível.

In [57]:
# ==============================================================================# 6 — Dashboard com KPI Cards (go.Indicator)# ==============================================================================ult = df_master['ano'].max()ant = ult - 1df_ult = df_master[df_master['ano'] == ult]df_ant = df_master[df_master['ano'] == ant]mort_ult = (df_ult['obitos'].sum() / df_ult['populacao'].sum()) * 100000mort_ant = (df_ant['obitos'].sum() / df_ant['populacao'].sum()) * 100000obt_total = df_ult['obitos'].sum()exames_total = df_ult['exames_realizados'].sum()cob_ult = (df_ult['exames_realizados'].sum() / df_ult['populacao'].sum()) * 100cob_ant = (df_ant['exames_realizados'].sum() / df_ant['populacao'].sum()) * 100fig_kpi = make_subplots(rows=1, cols=4,    specs=[[{'type':'indicator'}]*4],    subplot_titles=['Mortalidade/100k', 'Óbitos Totais', 'Exames Realizados', 'Cobertura (%)'])fig_kpi.add_trace(go.Indicator(    mode='number+delta', value=round(mort_ult, 2),    delta=dict(reference=round(mort_ant, 2), relative=True, valueformat='.1%'),    number=dict(font=dict(color='#E63946', size=36))), row=1, col=1)fig_kpi.add_trace(go.Indicator(    mode='number', value=obt_total,    number=dict(font=dict(color='#F4A261', size=36), valueformat=',')), row=1, col=2)fig_kpi.add_trace(go.Indicator(    mode='number', value=exames_total,    number=dict(font=dict(color='#2A9D8F', size=36), valueformat=',')), row=1, col=3)fig_kpi.add_trace(go.Indicator(    mode='number+delta', value=round(cob_ult, 2),    delta=dict(reference=round(cob_ant, 2), relative=True, valueformat='.1%'),    number=dict(font=dict(color='#457B9D', size=36), suffix='%')), row=1, col=4)fig_kpi.update_layout(    title=f'📊 Indicadores-Chave — {ult}',    height=250, paper_bgcolor='#1A1A2E',    font_color='#E0E0E0', title_font_size=18)fig_kpi.show()

## 7. Exportação & Reprodutibilidade> Exportar dados processados e garantir reprodutibilidade do projeto.

In [58]:
# ==============================================================================# 7 — Exportação de Dados# ==============================================================================# Salvar df_master finalsalvar_cache(df_master, 'df_master')# Salvar agregado por UFsalvar_cache(df_agg, 'df_agg_por_uf')print("\n📋 Arquivos no diretório data/:")for f in sorted(os.listdir(DATA_DIR)):    size = os.path.getsize(os.path.join(DATA_DIR, f))    print(f"  📄 {f} ({size/1024:.1f} KB)")print("\n✅ Projeto concluído! Todos os dados e visualizações estão prontos.")print("🔗 Para ver o dashboard interativo online:")print("   https://lookerstudio.google.com/reporting/fb59fc99-fc3f-43c1-a9dd-cf25f47ed02f/page/xwViF")